In [ ]:
import dataclasses
import logging
import shutil
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Iterable, Tuple

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.widgets  # Cursor


import seaborn as sns


import numpy as np
import pandas as pd
import polars as pl
import pprintpp
import scipy.integrate
import scipy.ndimage.interpolation


from corona_model import PROJECT_DIR, log_pth
from corona_model.countryinfo import CountryInfo
from corona_model.params import DiseaseParams, SimOpts, PlotOpts
from corona_model.world_data import CovidData


### Read in TLGRF dataset to obtain `r_t,c` for Colorado counties

In [ ]:
benchmark_TLGRF = pl.read_csv("benchmark_TLGRF_dataset.csv", low_memory=False)
benchmark_TLGRF = benchmark_TLGRF.with_columns(
    pl.col("date").str.strptime(pl.Date, fmt="%Y-%m-%d")
)

colorado_TLGRF = benchmark_TLGRF.filter(pl.col("state")=="Colorado")

In [ ]:
colorado_TLGRF.schema

In [ ]:
colorado_rtc = colorado_TLGRF.select(["fips","days_from_start","r_TLGRF","county","date"])
colorado_rtc.schema

In [ ]:
### First and last days where r_tc is non null for every county

In [ ]:
result = colorado_rtc.filter(pl.col("r_TLGRF").is_not_null())\
    .groupby("fips")\
    .agg(
        pl.max("days_from_start").alias("first_day"),
        pl.min("days_from_start").alias("last_day"),
    )

print(result)
print(result.max())

In [ ]:
logger = logging.getLogger(__name__)
logging.getLogger("matplotlib").setLevel(logging.INFO)


In [ ]:
test_fips = 8001

test_df = colorado_TLGRF.filter(pl.col("fips")==8001)
# Step 1: Sort the data by 'days_from_start'
test_df = test_df.sort("days_from_start")

# Step 2: Filter out leading nulls
# Find the first non-null index
first_non_null_index = test_df.filter(pl.col('r_TLGRF').is_not_null())['days_from_start'][0]
test_df = test_df.filter(pl.col('days_from_start') >= first_non_null_index)

# Step 3: Filter out trailing nulls
# Find the last non-null index
last_non_null_index = test_df.filter(pl.col('r_TLGRF').is_not_null())['days_from_start'][-1]
test_df = test_df.filter(pl.col('days_from_start') <= last_non_null_index)

# Step 4: Forward fill the intermediate null values
# Use the forward_fill function to fill nulls in 'r_TLGRF'
test_df = test_df.with_columns(
    pl.col('r_TLGRF').forward_fill().alias('r_TLGRF')
)

In [ ]:
test_df.head()

In [ ]:
days_list = test_df.select("days_from_start").to_pandas().values[:,0]
n_days = days_list[-1] - days_list[0] + 1
days_list, n_days

### Params and Relation to Growth Rate

Let the exponential growth rate be `r`. 
From the [Ma 2020](https://www.sciencedirect.com/science/article/pii/S2468042719300491) paper (they use `lambda` instead)


 - `r = (-(sigma + gamma) + ((sigma-gamma)**2+4*sigma*beta)**0.5)/2`

Therefore: `beta = ((r*2 + (sigma + gamma))**2 - (sigma - gamma)**2) / (4*sigma)` 
 
 - `SIR: R0 = r/gamma + 1`
 - `SEIR: R0 = (r+gamma)*(r+sigma)/(sigma*gamma)`

In [ ]:
gamma = 1.0 / (10 + 3)
        
# The rate at which an exposed person becomes infective.
# Source:
sigma = 1.0 / (5 - 3)    

test_rtc = test_df.select('r_TLGRF').to_pandas().reset_index().values

test_beta = test_df.select('r_TLGRF').to_pandas().reset_index().values
test_beta[:, 0] = test_beta[:, 0].astype(int)
test_beta[:,1] = ((test_beta[:,1]*2 + (sigma + gamma))**2 - (sigma - gamma)**2) / (4*sigma)

test_R0 = np.zeros(np.shape(test_rtc))
test_R0[:,1] = (test_rtc[:,1]+gamma)*(test_rtc[:,1]+sigma)/(sigma*gamma)

### Define Params

In [ ]:
@dataclass()
class DiseaseParams:
    """
    Disease parameters. Model is VERY sensitive to these, so they must be picked carefully from
    good sources.
    """
    # Gamma rate an infected recovers and moves into the recovered phase.
    # Source:
    gamma: float = 1.0 / (10 + 3)
        
    # The rate at which an exposed person becomes infective.
    # Source:
    sigma: float = 1.0 / (5 - 3)    
    
    test_beta = test_df.select('r_TLGRF').to_pandas().reset_index().values
    test_beta[:, 0] = test_beta[:, 0].astype(int)
    test_beta[:,1] = ((test_beta[:,1]*2 + (sigma + gamma))**2 - (sigma - gamma)**2) / (4*sigma)

    # Step 2: Convert the array to a list of tuples
    tuples_test_beta = [tuple(row) for row in test_beta]

    # Step 3: Convert the list of tuples to a tuple of tuples
    tuples_test_beta = tuple(tuples_test_beta)

    
    # Beta controls how often a susceptible-infected contact results in a new infection.
    # Source:
    beta_init = test_beta[0,1]
    
    beta: Tuple[Tuple[int, float]] = tuples_test_beta


    

    # TODO Check R values

    time_hospital: int = 10  # Days in hospital, after which patient either recovers or dies
    time_infected: int = 1.0 / gamma

    lag_communication: int = 0
    lag_testing: int = 2
    lag_symptom_to_hosp: int = 1

    rate_icu: float = 0.02  # Proportion of infected patients who end up in ICU
    rate_fatality_0: float = 0.008  # CFR of patients that 'recover' - either dead or alive
    rate_fatality_1: float = rate_fatality_0 * 2  # CFR once ICU beds are saturated  #TODO Need source

    frac_asymptomatic: float = 0.0  # Fraction of infected that don't show symptoms
    find_ratio: float = (1 - frac_asymptomatic)  # Proportion of infected which are found

    # r0: Tuple[Tuple[int, float]] = ((lock_del, b/gamma) for lock_del, b in beta)



@dataclass
class SimOpts:
    """ Simulation Options"""
    sim_length: int = 200  # In days
    lockdown: bool = True  # If True, a lockdown will be simulated by changing beta
    icu_beds: int = 4000  # ICU units available
    # hosp_beds: int = 0  # TODO Use this
    real_data_offset: int = 0  # How many days will the real world country data be delayed in the model
    initial_exposed: int = 26  # Number of initially exposed people
    add_delays: bool = True  # If True, will add delays to found cases, hospitalised, and deaths based on lags in DiseaseParams


@dataclass
class PlotOpts:
    plot_log: bool = True  # If true, plots will have a log y axis


### Define Simulations

In [ ]:
@dataclass
class SimResults:
    """ Class to store results"""
    T: np.ndarray  # Time-step (days), vector variable of ODEs
    S: np.ndarray  # Susceptible at each timestep
    E: np.ndarray  # Exposed at each timestep
    I: np.ndarray  # Infected at each timestep
    R: np.ndarray  # Recovered at each timestep
    D: np.ndarray = np.zeros(0)  # Deaths at each timestep
    F: np.ndarray = np.zeros(0)  # Found at each timestep
    H: np.ndarray = np.zeros(0)  # Hospitalised at each timestep
    P: np.ndarray = np.zeros(0)  # Probability of random person being infected at each timestep

### SEIR Object

In [ ]:
class SEIRModel(object):
    """ A SEIR model of the COVID-19 including ICU Saturation and testing delays."""

    def __init__(self, country_name: str):
        """
        :param country_name: Full name of country, e.g. "Australia"
        """
        #self.country = CountryInfo(country_name)
        #self.n_pop = self.country.population()
        self.country = "Adams"
        self.n_pop = 519883

    def model_seir(self, t: float, state: Iterable[np.ndarray], d_params: DiseaseParams,
                   s_opts: SimOpts) -> Tuple[float, float, float, float]:
        """
        Definition of SEIR model
        :param t: Time-step (days), dependant variable of ODEs
        :param state: Vector of ODE State variables [S, E, I, R]
        :param d_params: DiseaseParams dataclass from params, or your own/modified version
        :param s_opts: SimOpts dataclass from params, or your own/modified version
        :returns: 4-element tuple of change in each of the state variables
        """
        N = self.n_pop  # Population of country
        S, E, I, R = state

        beta = d_params.beta_init
        # Note we will use the lockdown flag to make beta time varying
        # beta time varying will allow for r_tc time varying
        if s_opts.lockdown is True:
            for lock_del, beta_val in d_params.beta:
                if t >= lock_del:
                    beta = beta_val

        sigma = d_params.sigma
        gamma = d_params.gamma

        dS = - beta * S * I / N
        dE = beta * S * I / N  - sigma * E
        dI = sigma * E - gamma * I
        dR = gamma * I
        return dS, dE, dI, dR

    def run_model(self, d_params: DiseaseParams, s_opts: SimOpts) -> SimResults:
        """
        Solves the ODE model and returns results.
        :param d_params: d_params: DiseaseParams dataclass from params, or your own/modified version
        :param s_opts: s_opts: SimOpts dataclass from params, or your own/modified version
        :returns: 5-element Tuple of arrays of results
        """
        T = np.arange(s_opts.sim_length)  # time-step Array
        Y0 = [self.n_pop - s_opts.initial_exposed, s_opts.initial_exposed, 0, 0]  # S, E, I, R at initial step

        logger.info(f"Starting run of model...")
        logger.info(f"DiseaseParams : {pprintpp.pformat(dataclasses.asdict(disease_params))}")
        logger.info(f"SimOpts : {pprintpp.pformat(dataclasses.asdict(sim_opts))}")
        logger.info(f"Initial conditions (S E I R): {Y0}")

        Y_RESULTS = scipy.integrate.solve_ivp(self.model_seir, t_span=[T[0], T[-1]],
                                              y0=Y0, args=(d_params, s_opts),
                                              t_eval=T)

        S, E, I, R = Y_RESULTS.y  # transpose and unpack

        logger.info(f"Solve complete!")
        raw_results = SimResults(T=T, S=S, E=E, I=I, R=R)
        parsed_results = self._parse_results(raw_results, d_params, s_opts)
        logger.info(f"Results successfully parsed.")
        return parsed_results

    def _parse_results(self, res: SimResults, d_params: DiseaseParams, s_opts: SimOpts) -> SimResults:
        """
        Parses the results created in run_model()
        :param res: SimResults from a model run
        :param d_params: d_params: DiseaseParams dataclass from params, or your own/modified version
        :param s_opts: s_opts: SimOpts dataclass from params, or your own/modified version
        :returns:
        """
        T, S, E, I, R = res.T, res.S, res.E, res.I, res.R
        F = I * d_params.find_ratio
        H = I * d_params.rate_icu * d_params.time_hospital / d_params.time_hospital
        P = I / self.n_pop * 1000000  # Probability of random person to be infected

        # estimate deaths from recovered
        D = np.arange(s_opts.sim_length)
        RPrev = 0
        DPrev = 0
        for i, t in enumerate(res.T):
            IFR = d_params.rate_fatality_0 if H[i] <= s_opts.icu_beds else d_params.rate_fatality_1
            D[i] = DPrev + IFR * (R[i] - RPrev)
            RPrev = R[i]
            DPrev = D[i]

        if s_opts.add_delays is True:
            F = self.delay(F, d_params.lag_symptom_to_hosp + d_params.lag_testing + d_params.lag_communication)
            H = self.delay(H, d_params.lag_symptom_to_hosp)  # ICU  from I
            D = self.delay(D, d_params.time_hospital + d_params.lag_communication)  # deaths  from R

        # Update result object
        res.P = P
        res.F = F
        res.H = P
        res.D = D

        return res

    @staticmethod
    def delay(arr: np.ndarray, days: int):
        return scipy.ndimage.interpolation.shift(arr, days, cval=0)


In [ ]:
test_model = SEIRModel("USA")

### Params and Relation to Growth Rate

Let the exponential growth rate be `r`. 
From the [Ma 2020](https://www.sciencedirect.com/science/article/pii/S2468042719300491) paper (they use `lambda` instead)


 - `r = (-(sigma + gamma) + ((sigma-gamma)**2+4*sigma*beta)**0.5)/2`

Therefore: `beta = ((r*2 + (sigma + gamma))**2 - (sigma - gamma)**2) / (4*sigma)` 
 
 - `SIR: R0 = r/gamma + 1`
 - `SEIR: R0 = (r+gamma)*(r+sigma)/(sigma*gamma)`

In [ ]:
disease_params = DiseaseParams()
sim_opts = SimOpts()
plot_opts = PlotOpts()

# Over-ride any options here on the fly, check params.py for all parameters!
sim_opts.sim_length = n_days
sim_opts.real_data_offset = 0  # How many days will the real world country data be delayed in the model

sim_opts.add_delays = True  # If True, will add delays to found cases, hospitalised, and deaths based on lags in DiseaseParams
sim_opts.lockdown = True  # If True, a lockdown will be simulated by changing beta

#beta_init = 1 / 2.5
#disease_params.beta = ((0, beta_init),
#                       (44, beta_init * 0.25),
#                       (47, beta_init * 0.24),
#                       (55, beta_init * 0.20))

plot_opts.plot_log = True  # If true, plots will have a log y axis

# Get real data and shift if required
#covid_data = CovidData()
#country_data = covid_data.world_data[country_info.iso(2)]

T = np.arange(n_days)  # time-step Array
#Y0 = [1000 - SimOpts.initial_exposed, SimOpts.initial_exposed, 0, 0]  # S, E, I, R at initial step

# Run model
test_model = SEIRModel("USA")
#test_model.model_seir(t=0, state=Y0, d_params=disease_params, s_opts=SimOpts)
results = test_model.run_model(disease_params, sim_opts)


In [ ]:
test_model.n_pop

### Remove JSON Read

In [ ]:
disease_params

In [ ]:
results

In [ ]:
#plt.plot(results.S/test_model.n_pop, label="S")
plt.plot(results.E, label="E")
plt.plot(results.I, label="I")
#plt.plot(results.R, label="R")
plt.plot(test_df.select("rolled_cases")[:,0].to_pandas().values, label="rolled_cases")
#plt.plot(test_df.select("log_rolled_cases")[:,0].to_pandas().values, label="log_rolled_cases")

plt.legend()
plt.show()

In [ ]:
plt.plot(test_rtc[:,1], label="r_tc")
plt.plot(test_beta[:,1], label="beta")
#plt.plot(test_R0[:,1], label="R0")
plt.legend()
plt.show()

In [ ]:
plt.plot(test_R0[:,1], label="R0")
plt.plot(np.ones(n_days))
plt.legend()
plt.show()

In [ ]:
plt.plot(1 - 1/test_R0[:,1], label="Herd Immunity")
plt.legend()
plt.show()

In [ ]:
np.exp(1)**max(test_df.to_pandas().log_rolled_cases)